In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append("../../")
import data_loading as dl
from data_loading import load_runs
from importlib import reload
reload(dl)

from microfit import run_plotter as rp
from microfit import histogram as hist

from microfit import variable_definitions as vdef
from microfit import selections

In [5]:
RUN = ["3"]
#RUN = ["1","2","3"]
#RUN = ["1","2","3_nocrt","3_crt","4b","4c","4d","5"]
#RUN = ["1","2","3","4b","4c","4d","5"]
        #important that it's a string 1) new format to include latest runs 2) to include 'mc_pdg' otherwise it gets dropped

rundata, mc_weights, data_pot = dl.load_runs(
    RUN,
    data="bnb",
    loadpi0variables=False,
    loadshowervariables=True,
    loadrecoveryvars=False,
    loadsystematics=True,
    numupresel=True,
    loadnumuvariables=True,
    use_bdt=False,
    load_lee=False,
    blinded=True,
    load_crt_vars=False,
    load_oLEE=True,
    enable_cache=True,
)

Loading run 3


/exp/uboone/app/users/mmoudgal/miniforge3/envs/python3LEE/lib/python3.7/site-packages/awkward/array/jagged.py:1043: RuntimeWarning: overflow encountered in power
  result = getattr(ufunc, method)(*inputs, **kwargs)
/exp/uboone/app/users/mmoudgal/miniforge3/envs/python3LEE/lib/python3.7/site-packages/pandas/core/generic.py:2505: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block5_values] [items->Index(['mc_pdg', 'mc_E', 'true_vel_vector', 'true_crit_vel_test_vector'], dtype='object')]

  encoding=encoding,
../../data_loading.py:792: RuntimeWarning: invalid value encountered in true_divide
  df["proton_pz"] = np.where((mc_E_prot > 0), mc_pz_prot / mc_p_prot, np.nan)
/exp/uboone/app/users/mmoudgal/miniforge3/envs/python3LEE/lib/python3.7/site-packages/pandas/core/generic.py:2505: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map 

In [11]:
RUNstr = ""       #Make an empty string
for run in RUN:
    RUNstr = RUNstr + run
print(RUNstr)

for key, df in rundata.items():
    if key!='data':        
        df['shr_energy_cali'] = df['shr_energy_cali'] * 1/0.83

3


In [6]:
#index_refrac = 1.4620 # Index of refraction for the Marcol 7 mineral oil used by MiniBooNE
#c = 1      #Natural units
#c1 = 3e8  #m/s
#c2 = 511940000000000 #This is the speed of light squared in MeV.
# Use either velocity formula v>sqrt(c2)/index_refrac or momentum formula m*sqrt(c2)<p*((index_refrac^2)-1)^(0.5)

# Cut?
# if pt > (m*sqrt(c2))/(((index_refrac^2)-1)^(0.5))

In [6]:
print("mc_pdg" in rundata["ext"].columns)

True


In [7]:
rundata["ext"]["mc_pdg"].head(20)

entry
0     []
1     []
2     []
3     []
4     []
5     []
6     []
7     []
8     []
9     []
10    []
11    []
12    []
13    []
14    []
15    []
16    []
17    []
18    []
19    []
Name: mc_pdg, dtype: object

In [ ]:
preselection = "None"
selection = "oLEETrue"

for binning_def in vdef.OLEE_variables:
    # some binning definitions have more than 4 elements,
    # we ignore the last ones for now
    binning = hist.Binning.from_config(*binning_def[:4])
    print(binning_def)
    signal_generator = hist.RunHistGenerator(
        rundata,
        binning,
        data_pot=data_pot,
        selection=selection,
        preselection=preselection,
        sideband_generator=None,
        uncertainty_defaults=None,
    )
    plotter = rp.RunHistPlotter(signal_generator)
    axes1 = plotter.plot(
        category_column="pdg_labels",
        include_multisim_errors=True,
        add_ext_error_floor=False,
        show_data_mc_ratio=False,
        show_chi_square=False,
    )
    
    plt.savefig(f'Plots/Research/{preselection}_{selection}_Run{RUNstr}_{binning_def[0]}.pdf', bbox_inches='tight')
    plt.savefig(f'Plots/Research/{preselection}_{selection}_Run{RUNstr}_{binning_def[0]}.png', bbox_inches='tight')
    plt.show()

#     axes2 = plotter.plot(
#         category_column="interaction",
#         include_multisim_errors=True,
#         add_ext_error_floor=False,
#         show_data_mc_ratio=False,
#         show_chi_square=False,
#     )

#     plt.savefig(f'plots/reco_study/int_{preselection}_{selection}_{binning_def[0]}.pdf', bbox_inches='tight')
#     plt.show()